In [ ]:
"""
KNN Hyperparameter Tuning: Random Search vs Bayesian Search
Dataset: CIC IoT 2023 (simulated with realistic class distribution)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
import time
from collections import defaultdict

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, ConfusionMatrixDisplay
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─────────────────────────────────────────────
# 1. SIMULATE CIC IoT 2023 DATASET
# ─────────────────────────────────────────────
print("=" * 65)
print("  KNN HYPERPARAMETER TUNING — CIC IoT 2023 Dataset")
print("=" * 65)

# Realistic IoT attack classes from CIC IoT 2023
CLASSES = [
    "Benign", "DDoS-ICMP_Flood", "DDoS-UDP_Flood", "DDoS-TCP_Flood",
    "DoS-UDP_Flood", "DoS-TCP_Flood", "Recon-PortScan",
    "Mirai-Botnet", "BruteForce-SSH", "MQTT-Malformed"
]

N_SAMPLES = 8000
N_FEATURES = 46   # realistic feature count for CIC IoT 2023

print(f"\n[*] Generating simulated CIC IoT 2023 data ...")
print(f"    Samples : {N_SAMPLES} | Features : {N_FEATURES} | Classes : {len(CLASSES)}")

# Simulate class-specific feature distributions
rng = np.random.default_rng(42)
X_list, y_list = [], []

class_weights = [0.30, 0.12, 0.10, 0.08, 0.08, 0.07, 0.07, 0.07, 0.06, 0.05]
class_counts  = [int(w * N_SAMPLES) for w in class_weights]
class_counts[-1] += N_SAMPLES - sum(class_counts)

for i, (cls, n) in enumerate(zip(CLASSES, class_counts)):
    mean = rng.uniform(-2, 2, N_FEATURES) * (i + 1) * 0.3
    cov  = np.eye(N_FEATURES) * rng.uniform(0.5, 2.0)
    X_cls = rng.multivariate_normal(mean, cov, n)
    X_list.append(X_cls)
    y_list.extend([cls] * n)

X = np.vstack(X_list)
y = np.array(y_list)

le = LabelEncoder()
y_enc = le.fit_transform(y)

# Train / test split (80/20 stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.20, stratify=y_enc, random_state=42
)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"    Train   : {X_train.shape[0]} | Test : {X_test.shape[0]}")

# ─────────────────────────────────────────────
# 2. HYPERPARAMETER SEARCH SPACE
# ─────────────────────────────────────────────
PARAM_SPACE = {
    "n_neighbors"   : list(range(1, 31)),
    "weights"       : ["uniform", "distance"],
    "metric"        : ["euclidean", "manhattan", "chebyshev", "minkowski"],
    "algorithm"     : ["auto", "ball_tree", "kd_tree", "brute"],
    "leaf_size"     : list(range(10, 51, 5)),
    "p"             : [1, 2, 3],
}

CV    = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
SCORE = "f1_macro"

def evaluate_params(params, X_tr, y_tr, cv=CV, scoring=SCORE):
    knn = KNeighborsClassifier(**params)
    scores = cross_val_score(knn, X_tr, y_tr, cv=cv, scoring=scoring, n_jobs=-1)
    return scores.mean()

# ─────────────────────────────────────────────
# 3. RANDOM SEARCH
# ─────────────────────────────────────────────
print("\n" + "─" * 65)
print("  [1] RANDOM SEARCH")
print("─" * 65)

N_RANDOM_ITER = 40
random_results = []
random_params  = []

t0 = time.time()
for i in range(N_RANDOM_ITER):
    params = {
        "n_neighbors" : int(rng.choice(PARAM_SPACE["n_neighbors"])),
        "weights"     : str(rng.choice(PARAM_SPACE["weights"])),
        "metric"      : str(rng.choice(PARAM_SPACE["metric"])),
        "algorithm"   : "auto",
        "leaf_size"   : int(rng.choice(PARAM_SPACE["leaf_size"])),
        "p"           : int(rng.choice(PARAM_SPACE["p"])),
    }
    score = evaluate_params(params, X_train, y_train)
    random_results.append(score)
    random_params.append(params)
    if (i + 1) % 10 == 0:
        print(f"    Iter {i+1:3d}/{N_RANDOM_ITER} | Best so far: {max(random_results):.4f}")

random_time   = time.time() - t0
best_rand_idx = np.argmax(random_results)
best_rand_par = random_params[best_rand_idx]
best_rand_sc  = random_results[best_rand_idx]

print(f"\n    Best F1  : {best_rand_sc:.4f}")
print(f"    Time     : {random_time:.1f}s")
print(f"    Params   : {best_rand_par}")

# ─────────────────────────────────────────────
# 4. BAYESIAN SEARCH (GP-based)
# ─────────────────────────────────────────────
print("\n" + "─" * 65)
print("  [2] BAYESIAN SEARCH  (Gaussian Process + EI acquisition)")
print("─" * 65)

# Encode categorical params numerically for GP
WEIGHTS_MAP    = {"uniform": 0, "distance": 1}
METRIC_MAP     = {"euclidean": 0, "manhattan": 1, "chebyshev": 2, "minkowski": 3}

def encode_params(params):
    return np.array([
        params["n_neighbors"] / 30.0,
        WEIGHTS_MAP[params["weights"]],
        METRIC_MAP[params["metric"]],
        (params["leaf_size"] - 10) / 40.0,
        (params["p"] - 1) / 2.0,
    ])

def decode_params(x):
    n_neighbors = max(1, min(30, int(round(x[0] * 30))))
    weights     = "distance" if x[1] > 0.5 else "uniform"
    metric_idx  = max(0, min(3, int(round(x[2]))))
    metric      = ["euclidean", "manhattan", "chebyshev", "minkowski"][metric_idx]
    leaf_size   = max(10, min(50, int(round(x[3] * 40 + 10))))
    p           = max(1, min(3, int(round(x[4] * 2 + 1))))
    return {"n_neighbors": n_neighbors, "weights": weights,
            "metric": metric, "algorithm": "auto",
            "leaf_size": leaf_size, "p": p}

def expected_improvement(X_new, X_obs, y_obs, gp, xi=0.01):
    mu, sigma = gp.predict(X_new, return_std=True)
    mu_best   = np.max(y_obs)
    Z         = (mu - mu_best - xi) / (sigma + 1e-9)
    ei        = (mu - mu_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

N_BAYES_INIT = 8
N_BAYES_ITER = 32
DIM          = 5

# Initial random points
bayes_X, bayes_y = [], []
t0 = time.time()

for _ in range(N_BAYES_INIT):
    x = rng.uniform(0, 1, DIM)
    x[1] = round(x[1])          # categorical: weights
    x[2] = round(x[2] * 3) / 3  # categorical: metric
    x[4] = round(x[4] * 2) / 2  # categorical: p
    p    = decode_params(x)
    sc   = evaluate_params(p, X_train, y_train)
    bayes_X.append(x)
    bayes_y.append(sc)

gp = GaussianProcessRegressor(
    kernel=Matern(nu=2.5),
    alpha=1e-6,
    normalize_y=True,
    n_restarts_optimizer=3
)

bayes_results = list(bayes_y)
bayes_params  = [decode_params(x) for x in bayes_X]

for i in range(N_BAYES_ITER):
    gp.fit(np.array(bayes_X), np.array(bayes_y))

    # Generate random candidate points
    candidates  = rng.uniform(0, 1, (2000, DIM))
    candidates[:, 1] = np.round(candidates[:, 1])
    candidates[:, 2] = np.round(candidates[:, 2] * 3) / 3
    candidates[:, 4] = np.round(candidates[:, 4] * 2) / 2

    ei_vals = expected_improvement(candidates, np.array(bayes_X), np.array(bayes_y), gp)
    best_c  = candidates[np.argmax(ei_vals)]

    p  = decode_params(best_c)
    sc = evaluate_params(p, X_train, y_train)

    bayes_X.append(best_c)
    bayes_y.append(sc)
    bayes_results.append(sc)
    bayes_params.append(p)

    if (i + 1) % 10 == 0:
        print(f"    Iter {i+1:3d}/{N_BAYES_ITER} | Best so far: {max(bayes_results):.4f}")

bayes_time    = time.time() - t0
best_bay_idx  = np.argmax(bayes_results)
best_bay_par  = bayes_params[best_bay_idx]
best_bay_sc   = bayes_results[best_bay_idx]

print(f"\n    Best F1  : {best_bay_sc:.4f}")
print(f"    Time     : {bayes_time:.1f}s")
print(f"    Params   : {best_bay_par}")

# ─────────────────────────────────────────────
# 5. FINAL EVALUATION
# ─────────────────────────────────────────────
print("\n" + "─" * 65)
print("  [3] FINAL MODEL EVALUATION ON TEST SET")
print("─" * 65)

def test_model(params, label):
    knn = KNeighborsClassifier(**params)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average="macro")
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_test, y_pred, average="macro")
    print(f"\n  [{label}]")
    print(f"    Accuracy  : {acc:.4f}")
    print(f"    F1 Macro  : {f1:.4f}")
    print(f"    Precision : {prec:.4f}")
    print(f"    Recall    : {rec:.4f}")
    return knn, y_pred, acc, f1, prec, rec

knn_rand, y_pred_rand, acc_r, f1_r, prec_r, rec_r = test_model(best_rand_par, "Random Search Best")
knn_bayes, y_pred_bay, acc_b, f1_b, prec_b, rec_b = test_model(best_bay_par,  "Bayesian Search Best")

# Also baseline (default KNN)
knn_base = KNeighborsClassifier()
knn_base.fit(X_train, y_train)
y_pred_base = knn_base.predict(X_test)
acc_base  = accuracy_score(y_test, y_pred_base)
f1_base   = f1_score(y_test, y_pred_base, average="macro")
prec_base = precision_score(y_test, y_pred_base, average="macro", zero_division=0)
rec_base  = recall_score(y_test, y_pred_base, average="macro")
print(f"\n  [Baseline KNN (default)]")
print(f"    Accuracy  : {acc_base:.4f}")
print(f"    F1 Macro  : {f1_base:.4f}")

# ─────────────────────────────────────────────
# 6. COMPREHENSIVE VISUALIZATION
# ─────────────────────────────────────────────
COLOR_RAND  = "#4C72B0"
COLOR_BAYES = "#DD8452"
COLOR_BASE  = "#8172B3"

fig = plt.figure(figsize=(22, 28), facecolor="#F8F9FA")
fig.suptitle(
    "KNN Hyperparameter Tuning — CIC IoT 2023 Dataset\n"
    "Random Search  vs  Bayesian Search",
    fontsize=18, fontweight="bold", y=0.98, color="#1a1a2e"
)

gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Plot 1: Convergence Curves ──────────────────────────────
ax1 = fig.add_subplot(gs[0, :2])

rand_best = np.maximum.accumulate(random_results)
bay_best  = np.maximum.accumulate(bayes_results)

ax1.plot(range(1, len(rand_best)+1), rand_best,
         color=COLOR_RAND, lw=2.5, marker="o", markersize=4, label="Random Search (best)")
ax1.plot(range(1, len(bay_best)+1),  bay_best,
         color=COLOR_BAYES, lw=2.5, marker="s", markersize=4, label="Bayesian Search (best)")
ax1.axhline(f1_base, color=COLOR_BASE, ls="--", lw=1.8, label=f"Baseline KNN ({f1_base:.4f})")

ax1.fill_between(range(1, len(rand_best)+1), rand_best, f1_base, alpha=0.15, color=COLOR_RAND)
ax1.fill_between(range(1, len(bay_best)+1),  bay_best,  f1_base, alpha=0.15, color=COLOR_BAYES)

ax1.set_title("Convergence Curves (Best F1 Macro over Iterations)", fontsize=13, pad=10)
ax1.set_xlabel("Iteration", fontsize=11)
ax1.set_ylabel("F1 Macro (5-fold CV)", fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_facecolor("#FFFFFF")

# ── Plot 2: Score Distribution per method ───────────────────
ax2 = fig.add_subplot(gs[0, 2])
bp = ax2.boxplot(
    [random_results, bayes_results],
    labels=["Random\nSearch", "Bayesian\nSearch"],
    patch_artist=True,
    notch=True,
    widths=0.4
)
bp["boxes"][0].set_facecolor(COLOR_RAND + "99")
bp["boxes"][1].set_facecolor(COLOR_BAYES + "99")
for median in bp["medians"]:
    median.set(color="black", linewidth=2)

ax2.set_title("Score Distribution\nAll Iterations", fontsize=13, pad=10)
ax2.set_ylabel("F1 Macro", fontsize=11)
ax2.grid(True, alpha=0.3, axis="y")
ax2.set_facecolor("#FFFFFF")

# ── Plot 3 & 4: Confusion Matrices ──────────────────────────
class_labels_short = [c.replace("DDoS-", "D-").replace("DoS-", "S-")
                       .replace("Recon-", "R-").replace("-Botnet", "")
                       .replace("BruteForce-", "BF-").replace("MQTT-", "MQTT-")
                       for c in le.classes_]

for ax_idx, (ax_gs, y_pred, title, color) in enumerate([
    (gs[1, :2], y_pred_rand,  "Confusion Matrix — Random Search Best",  COLOR_RAND),
    (gs[1, 2],  y_pred_bay,   "Confusion Matrix — Bayesian Search Best", COLOR_BAYES),
]):
    ax = fig.add_subplot(ax_gs)
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues" if color == COLOR_RAND else "Oranges")
    plt.colorbar(im, ax=ax, fraction=0.046)
    tick_marks = np.arange(len(le.classes_))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(class_labels_short, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(class_labels_short, fontsize=7)
    thresh = cm_norm.max() / 2.0
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax.text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm_norm[i,j] > thresh else "black", fontsize=6)
    ax.set_title(title, fontsize=11, pad=8)
    ax.set_xlabel("Predicted", fontsize=9)
    ax.set_ylabel("True", fontsize=9)
    ax.set_facecolor("#FFFFFF")

# ── Plot 5: Per-class F1 comparison ─────────────────────────
ax5 = fig.add_subplot(gs[2, :])
rpt_rand  = classification_report(y_test, y_pred_rand,  output_dict=True, zero_division=0)
rpt_bayes = classification_report(y_test, y_pred_bay,   output_dict=True, zero_division=0)
rpt_base  = classification_report(y_test, y_pred_base,  output_dict=True, zero_division=0)

cls_idx   = [str(i) for i in range(len(le.classes_))]
f1_rand_c  = [rpt_rand.get(k, {}).get("f1-score", 0)  for k in cls_idx]
f1_bay_c   = [rpt_bayes.get(k, {}).get("f1-score", 0) for k in cls_idx]
f1_base_c  = [rpt_base.get(k, {}).get("f1-score", 0)  for k in cls_idx]

x = np.arange(len(le.classes_))
w = 0.26
ax5.bar(x - w, f1_base_c,  w, label="Baseline KNN",    color=COLOR_BASE,  alpha=0.85)
ax5.bar(x,     f1_rand_c,  w, label="Random Search",   color=COLOR_RAND,  alpha=0.85)
ax5.bar(x + w, f1_bay_c,   w, label="Bayesian Search", color=COLOR_BAYES, alpha=0.85)
ax5.set_xticks(x)
ax5.set_xticklabels(class_labels_short, rotation=30, ha="right", fontsize=8.5)
ax5.set_title("Per-Class F1 Score — Baseline vs Random Search vs Bayesian Search", fontsize=13)
ax5.set_ylabel("F1 Score", fontsize=11)
ax5.legend(fontsize=10)
ax5.grid(True, alpha=0.3, axis="y")
ax5.set_facecolor("#FFFFFF")
ax5.set_ylim(0, 1.08)

# ── Plot 6: Summary metrics table ───────────────────────────
ax6 = fig.add_subplot(gs[3, :2])
ax6.axis("off")

methods   = ["Baseline KNN\n(default)", "Random Search\n(best)", "Bayesian Search\n(best)"]
accs      = [f"{acc_base:.4f}",  f"{acc_r:.4f}",  f"{acc_b:.4f}"]
f1s       = [f"{f1_base:.4f}",   f"{f1_r:.4f}",   f"{f1_b:.4f}"]
precs     = [f"{prec_base:.4f}", f"{prec_r:.4f}",  f"{prec_b:.4f}"]
recs      = [f"{rec_base:.4f}",  f"{rec_r:.4f}",   f"{rec_b:.4f}"]
times     = ["—",               f"{random_time:.1f}s",  f"{bayes_time:.1f}s"]
iters     = ["—",               str(N_RANDOM_ITER), str(N_BAYES_INIT + N_BAYES_ITER)]

col_labels = ["Method", "Accuracy", "F1 Macro", "Precision", "Recall", "Time", "Iterations"]
table_data = list(zip(methods, accs, f1s, precs, recs, times, iters))

tbl = ax6.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc="center",
    loc="center",
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)

for j in range(len(col_labels)):
    tbl[0, j].set_facecolor("#2C3E50")
    tbl[0, j].set_text_props(color="white", fontweight="bold")

row_colors = ["#EAF0FB", "#FEF0E6", "#EAF0FB"]
for i, rc in enumerate(row_colors, 1):
    for j in range(len(col_labels)):
        tbl[i, j].set_facecolor(rc)

# Highlight best values
best_col_vals = {
    1: (accs,  max),
    2: (f1s,   max),
    3: (precs, max),
    4: (recs,  max),
}
for col, (vals, fn) in best_col_vals.items():
    numeric = [float(v) for v in vals]
    best_v  = fn(numeric)
    for row, nv in enumerate(numeric, 1):
        if nv == best_v:
            tbl[row, col].set_facecolor("#D5F5E3")
            tbl[row, col].set_text_props(fontweight="bold")

ax6.set_title("Performance Summary", fontsize=13, fontweight="bold", pad=12)

# ── Plot 7: Hyperparameter importance (n_neighbors sweep) ───
ax7 = fig.add_subplot(gs[3, 2])
k_vals  = list(range(1, 21))
k_scores = []
for k in k_vals:
    knn_tmp = KNeighborsClassifier(n_neighbors=k,
                                    weights=best_bay_par["weights"],
                                    metric=best_bay_par["metric"])
    sc = cross_val_score(knn_tmp, X_train, y_train, cv=3, scoring=SCORE).mean()
    k_scores.append(sc)

ax7.plot(k_vals, k_scores, color=COLOR_BAYES, lw=2.5, marker="o", markersize=5)
ax7.axvline(best_bay_par["n_neighbors"], color="red", ls="--", lw=1.8,
            label=f"Best k={best_bay_par['n_neighbors']}")
ax7.set_title("Effect of k on F1 Score\n(Bayesian best other params)", fontsize=11)
ax7.set_xlabel("k (n_neighbors)", fontsize=10)
ax7.set_ylabel("F1 Macro (CV)", fontsize=10)
ax7.legend(fontsize=9)
ax7.grid(True, alpha=0.3)
ax7.set_facecolor("#FFFFFF")

plt.savefig("/mnt/user-data/outputs/knn_hyperparameter_tuning.png",
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
print("\n[✓] Figure saved to outputs/")

# ─────────────────────────────────────────────
# 7. PRINT FINAL SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  FINAL COMPARISON SUMMARY")
print("=" * 65)
print(f"\n  {'Method':<28} {'Accuracy':>10} {'F1 Macro':>10} {'Precision':>10} {'Recall':>10}")
print(f"  {'─'*28} {'─'*10} {'─'*10} {'─'*10} {'─'*10}")
print(f"  {'Baseline KNN (default)':<28} {acc_base:>10.4f} {f1_base:>10.4f} {prec_base:>10.4f} {rec_base:>10.4f}")
print(f"  {'Random Search Best':<28} {acc_r:>10.4f} {f1_r:>10.4f} {prec_r:>10.4f} {rec_r:>10.4f}")
print(f"  {'Bayesian Search Best':<28} {acc_b:>10.4f} {f1_b:>10.4f} {prec_b:>10.4f} {rec_b:>10.4f}")
print(f"\n  Random Search   →  Iterations: {N_RANDOM_ITER}  |  Time: {random_time:.1f}s")
print(f"  Bayesian Search →  Iterations: {N_BAYES_INIT+N_BAYES_ITER}  |  Time: {bayes_time:.1f}s")

print(f"\n  Best Random Search Params:")
for k, v in best_rand_par.items(): print(f"    {k:<15}: {v}")

print(f"\n  Best Bayesian Search Params:")
for k, v in best_bay_par.items(): print(f"    {k:<15}: {v}")

print("\n  [✓] Done.\n")